In [1]:
import pandas as pd
import statsmodels.api as sm
from scipy import stats

In [2]:
df=pd.read_excel(r"C:\Users\ASUS\OneDrive\Desktop\hr_data.xlsx")
df

,EmployeeID,ExperienceYears,PerformanceScore,TrainingHours,Gender_Code,Salary
0,1,5,85,40,1,4500
1,2,2,60,10,0,3200
2,3,8,92,55,1,6100
3,4,3,75,20,0,3800
4,5,10,98,80,1,7500
5,6,1,55,5,1,2900
6,7,4,78,25,0,4100
7,8,7,88,45,0,5800
8,9,6,81,35,1,5200
9,10,9,95,60,0,6900


In [3]:
print(df['Salary'].describe())
Q1=df['Salary'].quantile(0.25)
Q2=df['Salary'].quantile(0.50)
Q3=df['Salary'].quantile(0.75)
print(f"Q1: {Q1}")
print(f"Q2: {Q2}")
print(f"Q3: {Q3}")

count      15.000000
mean     5080.000000
std      1650.627586
min      2900.000000
25%      3750.000000
50%      4600.000000
75%      6200.000000
max      8200.000000
Name: Salary, dtype: float64
Q1: 3750.0
Q2: 4600.0
Q3: 6200.0


In [5]:
print("--Shapiro testi nəticəsi--")
stat, p_value=stats.shapiro(df['Salary'])
print(f"p_value:{p_value:.4f}")
if p_value>0.05:
    print("Data normal paylanıb. H0 hipotezi qəbul edilir.")
else:
    print("Data normal paylanmayıb. H0 hipotezi rədd edilir.")


print("\n""--T testi nəticəsi--")
kişilər=df[df['Gender_Code']==1]['Salary']
qadınlar=df[df['Gender_Code']==0]['Salary']
stat_t, p_val_t=stats.ttest_ind(kişilər, qadınlar)
print(f"P_value_t: {p_val_t:.4f}")
if p_val_t<0.05:
    print("Cinsiyyətlər arasında əhəmiyyətli maaş fərqi var.")
else:
    print("Cinsiyyətlər arasında maaş fərqi yoxdur.")


print( "\n","--Korelyasiya matrisi--")
korelyasiya_matrisi=df[['ExperienceYears', 'PerformanceScore', 'TrainingHours', 'Salary']].corr()
print(korelyasiya_matrisi['Salary'])

--Shapiro testi nəticəsi--
p_value:0.4505
Data normal paylanıb. H0 hipotezi qəbul edilir.

--T testi nəticəsi--
P_value_t: 0.2945
Cinsiyyətlər arasında maaş fərqi yoxdur.

 --Korelyasiya matrisi--
ExperienceYears     0.996383
PerformanceScore    0.941552
TrainingHours       0.976906
Salary              1.000000
Name: Salary, dtype: float64


### Korrelyasiya matrisinin yekun analitik şərhi
Aparılan korrelyasiya analizi nəticəsində məlum olmuşdur ki, işçilərin Maaşı (Salary) ilə onların Təcrübə illəri (ExperienceYears — 0.99),
Təlim saatları (TrainingHours — 0.97) və Performans balları (PerformanceScore — 0.94) arasında mükəmmələ yaxın, çox güclü müsbət xətti əlaqə
mövcuddur. Təcrübəsi artan, davamlı inkişaf edən (təlim saatlarını çoxaldan) və yüksək performans göstərən əməkdaşlar riyazi bir 
qanunauyğunluqla daha yüksək maaşla mükafatlandırılırlar. Hər üç göstəricinin maaşla olan korrelyasiyasının 0.94-dən yüksək olması eyni 
zamanda bu dəyişənlərin öz aralarında da güclü əlaqədə ola biləcəyinə işarə edir. Bu vəziyyət gələcəkdə qurulacaq reqressiya modellərində
Multikollinearlıq riski yarada biləcəyi üçün model mərhələsindən əvvəl mütləq nəzərə alınmalı və tənzimlənməlidir.

In [6]:
print("--OLS modeli--")
y=df['Salary']
x=df[['ExperienceYears', 'PerformanceScore', 'TrainingHours']]
x_sm=sm.add_constant(x)
model=sm.OLS(y, x_sm)
netice=model.fit()
print(netice.summary())

--OLS modeli--
                            OLS Regression Results                            
Dep. Variable:                 Salary   R-squared:                       0.994
Model:                            OLS   Adj. R-squared:                  0.992
Method:                 Least Squares   F-statistic:                     573.6
Date:                Thu, 04 Jun 2026   Prob (F-statistic):           2.32e-12
Time:                        19:17:09   Log-Likelihood:                -93.958
No. Observations:                  15   AIC:                             195.9
Df Residuals:                      11   BIC:                             198.7
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const             2913.92

### Reqressiya modelinin yekun analitik şərhi
Modelin Gücü:R^2 = 0.994 göstərir ki, maaş dəyişkənliyinin 99.2%-i bu 3 sərbəst dəyişən tərəfindən mükəmməl şəkildə izah olunur.
Əsas Faktor: 'ExperienceYears' modeldə yeganə statistik əhəmiyyətli (p < 0.05) dəyişəndir və hər əlavə təcrübə ili maaşı orta hesabla 
550.68 vahid artırır.
Multikollinearlıq Təhlükəsi: Performans və Təlim saatlarının p-value dəyərinin yüksək (p > 0.05) çıxması və yüksək Condition Number 
(1.39e+03), dəyişənlər arasında güclü Multikollinearlıq olduğunu və bir-birlərini təkrarladıqlarını sübut edir.